# Day 10: Text Classification — Bag of Words + Neural Network

**Goal:** Build a sentiment classifier — your first model that processes real text!

### What we'll build

A neural network that reads movie reviews and predicts: **positive (1)** or **negative (0)**.

```
"This movie was amazing!"  →  positive (1)
"Terrible. Waste of time." →  negative (0)
```

### The pipeline

```
Raw text  →  Tokenize  →  Bag of Words vector  →  Linear classifier  →  prediction
```

We'll cover:
1. Building a vocabulary from training data
2. Converting documents to BoW vectors
3. Training a linear classifier
4. Inspecting what the model learned (interpretability!)
5. Why BoW has limits (motivates Day 11: embeddings)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Our Dataset: Mini Movie Reviews

We'll use a small handcrafted dataset for transparency. (In production you'd use IMDb's 25,000-review dataset.)

In [ ]:
# A small but realistic sentiment dataset
# Positive reviews (label = 1) and negative reviews (label = 0)

reviews = [
    # Positive
    ("This movie was amazing! Best film I've seen all year.", 1),
    ("Wonderful acting and a beautiful story. Highly recommended.", 1),
    ("Loved every minute. The cinematography was stunning.", 1),
    ("Fantastic performances by the lead actors. Brilliant film.", 1),
    ("An incredible journey. The ending was perfect.", 1),
    ("Heartwarming and inspiring. A must-watch masterpiece.", 1),
    ("Hilarious from start to finish. Great writing and direction.", 1),
    ("The best film of the decade. Stunning visuals and great plot.", 1),
    ("Truly excellent. The acting was superb and the music wonderful.", 1),
    ("I absolutely loved this movie. So engaging and beautiful.", 1),
    ("Perfectly paced and brilliantly acted. A real gem.", 1),
    ("Outstanding work. The director did a fantastic job.", 1),
    ("This film is a masterpiece. Every scene is gorgeous.", 1),
    ("Phenomenal acting and a captivating story throughout.", 1),
    ("A beautiful and moving film. Truly inspiring.", 1),
    
    # Negative
    ("Terrible movie. Complete waste of time.", 0),
    ("Boring and predictable. The acting was awful.", 0),
    ("I hated every minute. Worst film I've ever seen.", 0),
    ("Awful direction and weak script. Skip this one.", 0),
    ("Painfully slow. The plot made no sense at all.", 0),
    ("Terrible acting and a stupid story. Avoid this film.", 0),
    ("So boring. I almost fell asleep watching it.", 0),
    ("A complete disaster. The worst movie of the year.", 0),
    ("Horrible writing. The actors looked confused throughout.", 0),
    ("Disappointing and dull. Not worth your time.", 0),
    ("I want my money back. This was a terrible experience.", 0),
    ("Cliched and badly acted. Skip this awful film.", 0),
    ("Truly bad. The plot was confusing and the acting weak.", 0),
    ("Painful to watch. Terrible direction and worse acting.", 0),
    ("Boring, predictable, awful. Do not waste your time.", 0),
]

print(f"Total reviews: {len(reviews)}")
print(f"Positive: {sum(1 for _, l in reviews if l == 1)}")
print(f"Negative: {sum(1 for _, l in reviews if l == 0)}")
print(f"\nSample positive: '{reviews[0][0]}'")
print(f"Sample negative: '{reviews[15][0]}'")

## 2. Tokenization — Split Text Into Words

For text classification, we tokenize at the **word level** (not character-level like Day 4).

A simple approach:
1. Lowercase everything (so "Movie" and "movie" become the same token)
2. Strip punctuation
3. Split on whitespace

In [ ]:
# A simple word-level tokenizer

def tokenize(text):
    """Lowercase, strip punctuation, split on whitespace."""
    text = text.lower()
    # Replace anything that's not a letter, digit, or apostrophe with a space
    text = re.sub(r"[^a-z0-9' ]+", " ", text)
    return text.split()

# Try it
sample = "This movie was amazing! Best film I've seen all year."
tokens = tokenize(sample)
print(f"Original: '{sample}'")
print(f"Tokens:   {tokens}")

## 3. Build the Vocabulary

Collect all unique words from the training data. Each word gets an index.

**Important:** Build the vocab from TRAIN data only. If you include test data, you're cheating (the model would know about words it shouldn't).

In [ ]:
# Split data: 80% train, 20% test
np.random.seed(0)
indices = np.random.permutation(len(reviews))
split = int(0.8 * len(reviews))
train_data = [reviews[i] for i in indices[:split]]
test_data = [reviews[i] for i in indices[split:]]

print(f"Train: {len(train_data)} reviews")
print(f"Test:  {len(test_data)} reviews\n")

# Count word frequencies in TRAIN data
word_counts = Counter()
for text, label in train_data:
    word_counts.update(tokenize(text))

print(f"Total unique words: {len(word_counts)}")
print(f"\nMost common words:")
for word, count in word_counts.most_common(15):
    print(f"  {word:>15}: {count}")

# Build vocab — include <unk> for unknown words at index 0
vocab = ['<unk>'] + [w for w, _ in word_counts.most_common()]
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for i, w in enumerate(vocab)}
vocab_size = len(vocab)

print(f"\nVocab size (including <unk>): {vocab_size}")

# Show a few sample lookups (use .get() to avoid KeyError if a word isn't in train)
sample_words = ['the', 'movie', 'great', 'terrible', 'amazing', 'banana']
print(f"\nSample word → index lookups:")
for w in sample_words:
    idx = word_to_idx.get(w)
    if idx is not None:
        print(f"  '{w}' → {idx}")
    else:
        print(f"  '{w}' → not in train vocab (would be treated as <unk>)")

## 4. Bag of Words — Convert Text to a Vector

Each document becomes a vector of length `vocab_size`, where position `i` = count of word `i`.

In [ ]:
# Convert a document to a Bag-of-Words vector

def text_to_bow(text, word_to_idx, vocab_size):
    """Turn text into a count vector of length vocab_size."""
    vec = torch.zeros(vocab_size, dtype=torch.float32)
    for token in tokenize(text):
        idx = word_to_idx.get(token, 0)   # 0 = <unk> for unknown words
        vec[idx] += 1
    return vec

# Try it on a sample
sample_text = "This movie was amazing!"
sample_vec = text_to_bow(sample_text, word_to_idx, vocab_size)

print(f"Text: '{sample_text}'")
print(f"BoW vector shape: {sample_vec.shape}")
print(f"Number of non-zero entries: {(sample_vec > 0).sum().item()}")

# Show which words got counted
print(f"\nWord counts in this document:")
nonzero = sample_vec.nonzero().squeeze().tolist()
if isinstance(nonzero, int):
    nonzero = [nonzero]
for idx in nonzero:
    word = idx_to_word[idx]
    count = sample_vec[idx].item()
    print(f"  {word:>15}: {int(count)}")

In [ ]:
# Convert ALL data to BoW vectors

X_train = torch.stack([text_to_bow(text, word_to_idx, vocab_size) for text, _ in train_data])
y_train = torch.tensor([label for _, label in train_data], dtype=torch.long)

X_test = torch.stack([text_to_bow(text, word_to_idx, vocab_size) for text, _ in test_data])
y_test = torch.tensor([label for _, label in test_data], dtype=torch.long)

print(f"X_train shape: {X_train.shape}  (samples × vocab_size)")
print(f"y_train shape: {y_train.shape}  (one label per sample)")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

## 5. Build the Classifier — Just a Linear Layer

For BoW, surprisingly, ONE linear layer works really well. Each weight learns "if word X appears, lean toward class Y."

We'll use:
- 2 output classes (positive, negative)
- Cross-Entropy loss (the right loss for multi-class classification)

In [ ]:
class BoWClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(vocab_size, num_classes)
    
    def forward(self, x):
        return self.fc(x)   # return logits (no softmax — CrossEntropyLoss does that)

# Build it
model = BoWClassifier(vocab_size=vocab_size, num_classes=2)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  (weights: vocab_size × num_classes = {vocab_size} × 2)")
print(f"  (bias:    num_classes = 2)")

## 6. Train It

In [ ]:
# Standard training loop

torch.manual_seed(42)
model = BoWClassifier(vocab_size, num_classes=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.CrossEntropyLoss()

train_losses = []
test_losses = []
test_accs = []

for epoch in range(150):
    # Train
    model.train()
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())
    
    # Evaluate on test set
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test)
        test_loss = loss_fn(test_logits, y_test)
        preds = test_logits.argmax(dim=1)
        acc = (preds == y_test).float().mean().item()
    test_losses.append(test_loss.item())
    test_accs.append(acc)
    
    if epoch % 30 == 0:
        print(f"Epoch {epoch:3d}: train_loss={loss:.4f}, test_loss={test_loss:.4f}, test_acc={acc:.2f}")

print(f"\nFinal test accuracy: {test_accs[-1]*100:.1f}%")

In [ ]:
# Plot training progress

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(train_losses, 'b-', linewidth=2, label='Train loss')
ax1.plot(test_losses, 'r-', linewidth=2, label='Test loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('Training Progress')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot([a * 100 for a in test_accs], 'g-', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Test Accuracy')
ax2.set_ylim(0, 105)
ax2.axhline(y=100, color='gray', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Inspect What the Model Learned

The cool part of BoW + Linear: we can directly look at the weights and see which words the model thinks are most positive/negative!

In [ ]:
# What does the model think each word means?

# weights shape: (num_classes, vocab_size) → (2, vocab_size)
# weights[0] = "vote" for negative class
# weights[1] = "vote" for positive class
weights = model.fc.weight.data           # shape (2, vocab_size)

# Sentiment score per word: positive vote - negative vote
sentiment = weights[1] - weights[0]      # shape (vocab_size,)

# Top 10 most "positive" and "negative" words
top_pos_idx = sentiment.topk(10).indices
top_neg_idx = (-sentiment).topk(10).indices

print("TOP 10 POSITIVE words (model thinks these → positive review):")
for idx in top_pos_idx:
    word = idx_to_word[idx.item()]
    score = sentiment[idx].item()
    print(f"  {word:>15}: {score:+.3f}")

print("\nTOP 10 NEGATIVE words (model thinks these → negative review):")
for idx in top_neg_idx:
    word = idx_to_word[idx.item()]
    score = sentiment[idx].item()
    print(f"  {word:>15}: {score:+.3f}")

In [ ]:
# Visualize the most "polarized" words

# Combine top positive and top negative for the plot
top_words = []
top_scores = []

for idx in top_pos_idx[:8]:
    top_words.append(idx_to_word[idx.item()])
    top_scores.append(sentiment[idx].item())

for idx in top_neg_idx[:8]:
    top_words.append(idx_to_word[idx.item()])
    top_scores.append(sentiment[idx].item())

# Sort by score for nice visualization
order = np.argsort(top_scores)
top_words = [top_words[i] for i in order]
top_scores = [top_scores[i] for i in order]

colors = ['red' if s < 0 else 'green' for s in top_scores]

plt.figure(figsize=(10, 6))
plt.barh(top_words, top_scores, color=colors, edgecolor='black')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Sentiment score (positive = green, negative = red)')
plt.title('What the BoW classifier learned')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("These weights ARE the model's understanding of sentiment.")
print("This is one of the few cases where ML is fully interpretable!")

## 8. Test on New Reviews

Let's predict sentiment for reviews the model has NEVER seen:

In [ ]:
# Predict sentiment for completely new reviews

def predict_sentiment(text, model):
    """Returns ('positive'/'negative', probability)."""
    model.eval()
    vec = text_to_bow(text, word_to_idx, vocab_size).unsqueeze(0)  # add batch dim
    with torch.no_grad():
        logits = model(vec)
        probs = F.softmax(logits, dim=1)
    pred_class = probs.argmax(dim=1).item()
    confidence = probs[0, pred_class].item()
    label = "positive" if pred_class == 1 else "negative"
    return label, confidence

# Test on new reviews
new_reviews = [
    "What an amazing film! Beautiful and inspiring.",
    "Terrible. Awful. Truly bad.",
    "I loved the acting. Great movie!",
    "Painfully boring. Skip this one.",
    "The film was wonderful. The acting was superb.",
    "Such a bad film. Worst I've seen.",
]

print(f"{'Prediction':>10} | {'Confidence':>10} | Review")
print("-" * 80)
for review in new_reviews:
    label, conf = predict_sentiment(review, model)
    print(f"  {label:>8} | {conf*100:>9.1f}% | '{review}'")

## 9. The Limits of BoW — When It Fails

BoW ignores word ORDER. Let's see how this hurts:

In [ ]:
# Tricky cases — same words, different meanings

tricky_reviews = [
    "This was not amazing.",                          # "not amazing" = negative
    "Not boring at all! Really enjoyable.",            # "not boring" = positive
    "The acting was good but the plot was terrible.",  # mixed signals
    "I hate to say it, but this film is amazing.",     # "hate" but positive
    "Not great, not good, just terrible.",             # all negative-leaning
]

print(f"{'Prediction':>10} | {'Confidence':>10} | Review")
print("-" * 80)
for review in tricky_reviews:
    label, conf = predict_sentiment(review, model)
    print(f"  {label:>8} | {conf*100:>9.1f}% | '{review}'")

print("\nProblems with BoW:")
print("  1. 'not amazing' vs 'amazing' — BoW sees one extra 'not', mostly same.")
print("  2. Word order doesn't matter — but it should!")
print("  3. Unknown words get treated as <unk> (random)")
print("  4. No notion of similarity — 'great' and 'amazing' are unrelated")
print("\nThis is why we need EMBEDDINGS (Day 11) — dense vectors that capture meaning.")

## 10. Compare with a Multi-Layer Network

Does adding hidden layers help? Let's try:

In [ ]:
# Multi-layer BoW classifier

class DeepBoWClassifier(nn.Module):
    def __init__(self, vocab_size, hidden=64, num_classes=2, dropout=0.3):
        super().__init__()
        self.fc1 = nn.Linear(vocab_size, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.fc3 = nn.Linear(hidden, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)

# Train both models, compare
def train_and_eval(model, epochs=150, lr=0.05):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    test_accs = []
    for _ in range(epochs):
        model.train()
        loss = loss_fn(model(X_train), y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        model.eval()
        with torch.no_grad():
            preds = model(X_test).argmax(dim=1)
            test_accs.append((preds == y_test).float().mean().item())
    return test_accs

torch.manual_seed(42)
linear_model = BoWClassifier(vocab_size, num_classes=2)
torch.manual_seed(42)
deep_model = DeepBoWClassifier(vocab_size, hidden=32)

linear_accs = train_and_eval(linear_model)
deep_accs = train_and_eval(deep_model)

plt.figure(figsize=(10, 5))
plt.plot([a*100 for a in linear_accs], 'b-', label=f'Linear (final {linear_accs[-1]*100:.0f}%)', linewidth=2)
plt.plot([a*100 for a in deep_accs], 'r-', label=f'Deep MLP (final {deep_accs[-1]*100:.0f}%)', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Test accuracy (%)')
plt.title('Linear vs Deep MLP on BoW')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

linear_params = sum(p.numel() for p in linear_model.parameters())
deep_params = sum(p.numel() for p in deep_model.parameters())
print(f"Linear model params:    {linear_params:,}")
print(f"Deep MLP model params:  {deep_params:,}  ({deep_params/linear_params:.1f}x more)")
print(f"\nFor BoW, more layers don't help much.")
print(f"Why? Because BoW already discards too much info (word order).")
print(f"To get better, we need a BETTER representation: embeddings (Day 11).")

---

## Exercises

1. **Stop word removal:** Add a list of common stop words ('the', 'a', 'is', 'and', 'of') and remove them in `tokenize()`. Does accuracy go up or down? Why?

2. **TF-IDF instead of counts:** Replace raw counts with TF-IDF (Term Frequency × Inverse Document Frequency). This down-weights very common words automatically.

3. **Try IMDb dataset:** Load the real IMDb dataset (25,000 reviews) using `datasets` library or torchtext. Does the model still work? Does training take longer?

4. **Bigrams:** Modify the tokenizer to also include 2-word phrases (bigrams). Now "not amazing" becomes its own token. Does this help with the tricky cases?

5. **Build a confusion matrix:** Compute predictions on the test set and create a 2x2 confusion matrix. Calculate precision, recall, and F1.

---

## Key Takeaways

### The text classification pipeline:

```
Raw text
   ↓ tokenize         (split into words)
List of tokens
   ↓ vocabulary lookup (word → integer index)
List of token IDs
   ↓ vectorize         (BoW: count vector of length vocab_size)
Bag-of-Words vector
   ↓ neural network    (Linear or MLP)
Logits
   ↓ softmax + argmax
Predicted class
```

### What you can do now:

- ✓ Tokenize raw text
- ✓ Build vocabulary, handle unknown words (`<unk>`)
- ✓ Convert text to numerical vectors (BoW)
- ✓ Train a classifier with Cross-Entropy loss
- ✓ Inspect what the model learned (interpretable!)
- ✓ Make predictions on new text

### What's missing:

| Limitation | Why It Matters | Fix |
|------------|---------------|-----|
| No word similarity | "good" and "great" → unrelated | Embeddings (Day 11) |
| Ignores word order | "not amazing" ≈ "amazing not" | Sequence models (Day 13+) |
| Sparse vectors | Vocab of 50k → vectors of length 50k | Embeddings + LSTM/Transformer |
| Unknown words break | New word → just `<unk>` | Subword tokenization (BPE, Day 21) |

### The journey continues:

```
Day 1-9:   Built networks, trained them, saved them                ✓
Day 10:    First text task — BoW + linear classifier               ✓
Day 11:    Word embeddings — finally learn that "good ≈ great"
Day 12:    Project — full sentiment classifier with embeddings
Day 13+:   Sequence models that respect word order
```

**Tomorrow:** Word embeddings — the breakthrough that made modern NLP possible.